In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv('/content/retail-orders-raw.csv')

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)

display(df.head())

Dataset loaded successfully!
Rows and columns: (12, 9)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [3]:
import os

print(os.listdir('/content'))

['.config', 'retail-orders-raw.csv', 'sample_data']


In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include='all').T)

Column names:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          12 non-null     object 
 1   order_date        11 non-null     object 
 2   customer_segment  12 non-null     object 
 3   city              11 non-null     object 
 4   category          12 non-null     object 
 5   quantity          12 non-null     object 
 6   unit_price        12 non-null     int64  
 7   discount_pct      11 non-null     float64
 8   payment_status    12 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 996.0+ bytes

Statistical summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,12,11,RT-1004,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_date,11,10,2026-01-07,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_segment,12,4,Student,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,11,7,Chennai,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,12,3,Learning Kit,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,12,5,1,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unit_price,12.0,NaN,NaN,NaN,1082.333333,318.614425,799.0,799.0,999.0,1499.0,1499.0
discount_pct,11.0,NaN,NaN,NaN,13.636364,30.748245,0.0,0.0,5.0,10.0,105.0
payment_status,12,5,Paid,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# 1. Completeness: missing values
print("Missing values:")
display(df.isnull().sum().to_frame("Missing Count"))

# 2. Uniqueness: duplicate records
print("Duplicate rows:", df.duplicated().sum())

# 3. Validity: invalid numeric values, if applicable
numeric_cols = df.select_dtypes(include=np.number).columns

print("Numeric column summary:")
display(df[numeric_cols].describe().T)

# 4. Consistency: inspect column names and data types
print("Data types:")
display(df.dtypes.to_frame("Data Type"))

# 5. Freshness: inspect available date columns
date_cols = [
    col for col in df.columns
    if "date" in col.lower()
]

for col in date_cols:
    dates = pd.to_datetime(df[col], errors="coerce")
    print(f"{col}: latest valid date =", dates.max())

Missing values:


,Missing Count
order_id,0
order_date,1
customer_segment,0
city,1
category,0
quantity,0
unit_price,0
discount_pct,1
payment_status,0


Duplicate rows: 1
Numeric column summary:


,count,mean,std,min,25%,50%,75%,max
unit_price,12.0,1082.333333,318.614425,799.0,799.0,999.0,1499.0,1499.0
discount_pct,11.0,13.636364,30.748245,0.0,0.0,5.0,10.0,105.0


Data types:


,Data Type
order_id,object
order_date,object
customer_segment,object
city,object
category,object
quantity,object
unit_price,int64
discount_pct,float64
payment_status,object


order_date: latest valid date = 2026-01-18 00:00:00


In [6]:
quality_report = pd.DataFrame({
    "Check": [
        "Total Rows",
        "Total Columns",
        "Missing Cells",
        "Duplicate Rows"
    ],
    "Result": [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ]
})

display(quality_report)

quality_report.to_csv(
    "data_quality_report.csv",
    index=False
)

print("Data-quality report created!")

,Check,Result
0,Total Rows,12
1,Total Columns,9
2,Missing Cells,3
3,Duplicate Rows,1


Data-quality report created!


In [7]:

import pandas as pd
import numpy as np

# 1. Completeness
completeness = (
    (1 - df.isnull().sum() / max(len(df), 1)) * 100
).round(2)

# 2. Uniqueness
duplicate_rows = int(df.duplicated().sum())

# 3. Validity: flag negative values in numeric columns
numeric_cols = df.select_dtypes(include="number").columns
negative_counts = (df[numeric_cols] < 0).sum()

# 4. Consistency: inspect repeated column values and types
data_types = df.dtypes.astype(str)

# 5. Freshness: check date columns
date_results = {}
for col in df.columns:
    if "date" in col.lower():
        parsed = pd.to_datetime(df[col], errors="coerce")
        date_results[col] = {
            "Invalid dates": int(
                (df[col].notna() & parsed.isna()).sum()
            ),
            "Latest date": str(parsed.max())
        }

print("COMPLETENESS (%)")
display(completeness.to_frame("Completeness %"))

print("DUPLICATE ROWS:", duplicate_rows)
print("NEGATIVE VALUES BY NUMERIC COLUMN")
display(negative_counts.to_frame("Negative Count"))

print("DATA TYPES")
display(data_types.to_frame("Data Type"))

print("DATE QUALITY")
print(date_results)

COMPLETENESS (%)


,Completeness %
order_id,100.00
order_date,91.67
customer_segment,100.00
city,91.67
category,100.00
quantity,100.00
unit_price,100.00
discount_pct,91.67
payment_status,100.00


DUPLICATE ROWS: 1
NEGATIVE VALUES BY NUMERIC COLUMN


,Negative Count
unit_price,0
discount_pct,0


DATA TYPES


,Data Type
order_id,object
order_date,object
customer_segment,object
city,object
category,object
quantity,object
unit_price,int64
discount_pct,float64
payment_status,object


DATE QUALITY
{'order_date': {'Invalid dates': 2, 'Latest date': '2026-01-18 00:00:00'}}


In [8]:

print("Dataset columns:")
for col in df.columns:
    print("-", col)

print("\nNumber of rows:", len(df))

Dataset columns:
- order_id
- order_date
- customer_segment
- city
- category
- quantity
- unit_price
- discount_pct
- payment_status

Number of rows: 12


In [10]:

# Display columns to identify the correct KPI fields
print("Available columns:")
print(df.columns.tolist())

# Display a preview of the dataset
display(df.head())

Available columns:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [11]:

# Create a detailed data-quality summary
quality_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Total Rows": len(df),
    "Missing Values": df.isnull().sum().values,
    "Missing Percentage": (
        df.isnull().mean().values * 100
    ).round(2),
    "Unique Values": df.nunique().values
})

display(quality_summary)

# Export the report
quality_summary.to_csv(
    "retail_data_quality_summary.csv",
    index=False
)

print("Data-quality summary exported successfully!")

,Column,Data Type,Total Rows,Missing Values,Missing Percentage,Unique Values
0,order_id,object,12,0,0.00,11
1,order_date,object,12,1,8.33,10
2,customer_segment,object,12,0,0.00,4
3,city,object,12,1,8.33,7
4,category,object,12,0,0.00,3
5,quantity,object,12,0,0.00,5
6,unit_price,int64,12,0,0.00,3
7,discount_pct,float64,12,1,8.33,5
8,payment_status,object,12,0,0.00,5


Data-quality summary exported successfully!


In [12]:

print("Your dataset columns:")
print(df.columns.tolist())

print("\nSample records:")
display(df.head(5))

Your dataset columns:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Sample records:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
